This notebook will allow the user to truncate or delete  the tables in the SQLite database before running the Database_Setup_n_Data_Ingestion notebook.  This is to ensure that data is not replicated for each processing run.  It takes approximately 5 minutes to complete.

In [1]:
# Import modules
import os
from os.path import exists
import pandas as pd
import sqlite3
import time

In [2]:
# Define the path for the SQLite database
cwd = os.getcwd()
database_path = f'{cwd}/data/crash_data.db'

In [3]:
def check_table_exists(database_path, table_name):
    query = f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';"
    with sqlite3.connect(database_path) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        result = cursor.fetchone()
    return result is not None

In [4]:
def truncate_table(database_path, table_name, retries=5, delay=1):
    query = f"DELETE FROM {table_name};"
    for attempt in range(retries):
        try:
            with sqlite3.connect(database_path) as conn:
                cursor = conn.cursor()
                cursor.execute(query)
                conn.commit()
            break
        except sqlite3.OperationalError as e:
            if "database is locked" in str(e):
                print(f"Database is locked. Retry {attempt + 1}/{retries} after {delay} seconds.")
                time.sleep(delay)
            else:
                raise

In [5]:
def check_table_has_data(database_path, table_name):
    query = f"SELECT COUNT(*) FROM {table_name};"
    with sqlite3.connect(database_path) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        count = cursor.fetchone()[0]
    return count > 0

In [6]:
def delete_table(database_path, table_name, retries=3, delay=1):
    query = f"DROP TABLE {table_name};"
    for attempt in range(retries):
        try:
            with sqlite3.connect(database_path) as conn:
                cursor = conn.cursor()
                cursor.execute(query)
                conn.commit()
            break
        except sqlite3.OperationalError as e:
            if "database is locked" in str(e):
                print(f"Database is locked. Retry {attempt + 1}/{retries} after {delay} seconds.")
                time.sleep(delay)
            else:
                raise

In [7]:
# Connect to SQLite database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

table_names = ['ksp_incidents', 'ksp_controls', 'ksp_person', 'ksp_vehicles', 'ksp_factors', 'county_district_lut', 'unit_factor_lut']

for table_name in table_names:
    if check_table_exists(database_path, table_name):
        print(f"The table '{table_name}' exists.")
        truncate_table(database_path, table_name)
        if check_table_has_data(database_path, table_name):
            print(f"The table '{table_name}' still has data after truncation.")
            delete_table(database_path, table_name)
            print(f"The table '{table_name}' has been deleted.")
        else:
            print(f"The table '{table_name}' is empty after truncation.")
    else:
        print(f"The table '{table_name}' does not exist.")

The table 'ksp_incidents' exists.
The table 'ksp_incidents' is empty after truncation.
The table 'ksp_controls' exists.
The table 'ksp_controls' is empty after truncation.
The table 'ksp_person' exists.
The table 'ksp_person' is empty after truncation.
The table 'ksp_vehicles' exists.
The table 'ksp_vehicles' is empty after truncation.
The table 'ksp_factors' exists.
The table 'ksp_factors' is empty after truncation.
The table 'county_district_lut' exists.
The table 'county_district_lut' is empty after truncation.
The table 'unit_factor_lut' does not exist.
